# 03. GANs: DCGAN, WGAN-GP, and StyleGAN Concepts

- **Category:** 05_architectures / 05_generative_models
- **Lineage & Citations:** Ian Goodfellow et al., 2014 (*GAN*); Alec Radford et al., 2015 (*DCGAN*); Ishaan Gulrajani et al., 2017 (*WGAN-GP*); Tero Karras et al., 2019/2020 (*StyleGAN1/2*).
- **Purpose:** Formulates the minimax game between Generator and Discriminator, Earth Mover's Wasserstein distance with gradient penalty, and StyleGAN2 weight modulation and demodulation.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Neural Audio Codecs Encodec RVQ and Quantizer Dropout](02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.ipynb) $\to$ `🔗 Downstream Syntheses:` [Normalizing Flows RealNVP Glow and Spline Flows](04_normalizing_flows_realnvp_glow_and_spline_flows.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**
> - **BatchNorm in WGAN-GP Critics:** Cross-sample batch statistics destroy the per-sample 1-Lipschitz gradient norm constraint.
> - **Weight Clipping in WGAN:** Weight clamping to $[-c, c]$ causes gradient vanishing and capacity collapse; use gradient penalty instead.

## Mathematical Foundations & Governing Equations

### 1. WGAN-GP Critic Objective
$$\mathcal{L}_{\text{critic}} = \mathbb{E}[D(\tilde{x})] - \mathbb{E}[D(x)] + \lambda \mathbb{E}\left[(\|\nabla_{\hat{x}} D(\hat{x})\|_2 - 1)^2\right]$$

### 2. StyleGAN2 Weight Modulation & Demodulation
Modulate: $w'_{i, j} = s_j \cdot w_{i, j}$. Demodulate:
$$w''_{i, j} = \frac{w'_{i, j}}{\sqrt{\sum_j (w'_{i, j})^2 + \epsilon}}$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/05_generative_models/03_gans_dcgan_wgan_gp_and_stylegan_concepts.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

x_adain = weights["x_adain"]
scale = weights["scale"]
bias_adain = weights["bias_adain"]
x_feat = weights["x_feat"]
w_conv = weights["w_conv"]
s_style = weights["s_style"]
print(f"Loaded {len(weights)} golden tensors on {env.device}, x_adain: {list(x_adain.shape)}, w_conv: {list(w_conv.shape)}")

In [ ]:
def adain(x, scale, bias, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    norm = (x - mean) / torch.sqrt(var + eps)
    return norm * scale + bias

def stylegan2_weight_demodulation(x, weights, styles, eps=1e-8):
    w_mod = weights.unsqueeze(0) * styles.unsqueeze(1)
    demod_factors = torch.rsqrt(w_mod.pow(2).sum(dim=-1, keepdim=True) + eps)
    w_demod = w_mod * demod_factors
    return (w_demod @ x.unsqueeze(-1)).squeeze(-1)

def wgan_critic_loss(real_scores, fake_scores):
    return fake_scores.mean() - real_scores.mean()

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
adain_res = adain(x_adain, scale, bias_adain)
err_adain = float(torch.max(torch.abs(adain_res - weights["adain_out"])))
assert err_adain < 1e-6, f"AdaIN error: {err_adain:.2e}"

demod_res = stylegan2_weight_demodulation(x_feat, w_conv, s_style)
err_demod = float(torch.max(torch.abs(demod_res - weights["demod_out"])))
assert err_demod < 1e-6, f"Demod error: {err_demod:.2e}"

wgan_res = wgan_critic_loss(weights["real_scores"], weights["fake_scores"])
err_wgan = float(torch.max(torch.abs(wgan_res - weights["wgan_loss"][0])))
assert err_wgan < 1e-6, f"WGAN loss error: {err_wgan:.2e}"
print(f"Parity Verified: AdaIN={err_adain:.2e}, Demod={err_demod:.2e}, WGAN Loss={err_wgan:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_adain = benchmark_fn(lambda: adain(x_adain, scale, bias_adain), iters=50, name="AdaIN Normalization")
b_demod = benchmark_fn(lambda: stylegan2_weight_demodulation(x_feat, w_conv, s_style), iters=50, name="Weight Demodulation")
print(format_benchmark_table([b_adain, b_demod]))

# Visualization: StyleGAN2 Demodulated Weight Heatmap
w_mod_ex = (w_conv.unsqueeze(0) * s_style.unsqueeze(1))[0].detach().numpy()
w_demod_ex = stylegan2_weight_demodulation(x_feat, w_conv, s_style).detach().numpy()

fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
im0 = ax[0].imshow(w_mod_ex, cmap="coolwarm", aspect="auto")
ax[0].set_title("Modulated Weights W' [C_out, C_in]")
plt.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(w_demod_ex, cmap="viridis", aspect="auto")
ax[1].set_title("Demodulated Features Y [B, C_out]")
plt.colorbar(im1, ax=ax[1])
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse vs Corrected Implementation: Weight Clipping vs Gradient Penalty
# Weight clipping forces weights to +/- c, killing expressivity.
# Gradient penalty enforces per-sample Lipschitz continuity smoothly.
c = 0.05
clipped_w = torch.clamp(w_conv.clone(), -c, c)
sat_ratio = float((clipped_w.abs() >= c - 1e-5).float().mean().item())
print(f"Weight Clipping Saturated Fraction (|w| >= {c}): {sat_ratio:.2%}")
assert sat_ratio > 0.0, "Weight clipping causes artificial parameter saturation at boundaries!"
print("Demonstrated requirement of soft Gradient Penalty over hard weight clipping.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Kantorovich Duality Over Jensen-Shannon:** WGAN provides meaningful gradients everywhere, even when distributions do not overlap.
2. **Direct Weight Demodulation:** Normalizing weights rather than activations avoids water droplet artifacts and stabilizes high-resolution synthesis.
3. **Per-Sample Gradient Penalty:** Enforces the Lipschitz continuity constraint without restricting neural architecture or weight bounds.

---
### Cross-Domain Generalisation Matrix
- **High-Fidelity Neural Rendering:** StyleGAN architectures form the backbone for controllable 3D neural avatars and radiance fields.
- **Image-to-Image Domain Translation:** CycleGAN and Pix2Pix translate between distinct image modalities without paired supervision.
- **Speech and Audio Synthesis:** HiFi-GAN and BigVGAN apply multi-period discriminator formulations for high-fidelity neural vocoding.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_gans_dcgan_wgan_gp_and_stylegan_concepts.md](../../../markdown/05_architectures/05_generative_models/03_gans_dcgan_wgan_gp_and_stylegan_concepts.md)
- **PyTorch 2.x Module:** [03_gans_dcgan_wgan_gp_and_stylegan_concepts.py](../../../pytorch2/05_architectures/05_generative_models/03_gans_dcgan_wgan_gp_and_stylegan_concepts.py)
- **Functional JAX Module:** [03_gans_dcgan_wgan_gp_and_stylegan_concepts.py](../../../jax/05_architectures/05_generative_models/03_gans_dcgan_wgan_gp_and_stylegan_concepts.py)
- **Candle Rust Module:** [03_gans_dcgan_wgan_gp_and_stylegan_concepts.rs](../../../candle/05_architectures/05_generative_models/03_gans_dcgan_wgan_gp_and_stylegan_concepts.rs)
- **Hardware Kernel:** [03_gans_dcgan_wgan_gp_and_stylegan_concepts.rs](../../../kernels/05_architectures/05_generative_models/03_gans_dcgan_wgan_gp_and_stylegan_concepts.rs)
- **Universal Golden SafeTensors:** [03_gans_dcgan_wgan_gp_and_stylegan_concepts.safetensors](../../../weights/05_architectures/05_generative_models/03_gans_dcgan_wgan_gp_and_stylegan_concepts.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 02. Neural Audio Codecs Encodec RVQ and Quantizer Dropout](02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 04. Normalizing Flows RealNVP Glow and Spline Flows](04_normalizing_flows_realnvp_glow_and_spline_flows.ipynb) &rarr;